# Practice on AQE (Adaptive Query Execution)

A hands-on tour of Spark's Adaptive Query Execution: what it is, the key optimizations it performs at runtime (shuffle-partition coalescing, join-strategy switching, skew-join splitting), and runnable demos with skewed data.

## What is AQE?

Classic Spark query execution is **static**: the whole physical plan is decided once, upfront, based on *estimates* (table statistics, row counts, data sizes). Those estimates are often wrong — especially after filters and joins — which leads to plans that shuffle too much, pick the wrong join strategy, or leave some tasks doing 99% of the work.

**Adaptive Query Execution** (Spark 3.0+, enabled by default since Spark 3.2 and on all recent Databricks runtimes) re-optimizes the query **at runtime**, at each shuffle boundary, using **actual observed statistics** (real row counts and sizes per partition). Spark pauses at the shuffle, measures the data, and can rewrite the remaining plan.

The three headline optimizations:

1. **Dynamically coalescing shuffle partitions** — merge many tiny post-shuffle partitions into fewer, right-sized ones (target size `spark.sql.adaptive.advisoryPartitionSizeInBytes`).
2. **Dynamically switching join strategies** — convert a planned sort-merge/shuffle-hash join into a **broadcast hash join** once actual runtime stats show one side is small enough (limit `spark.sql.adaptive.autoBroadcastJoinThreshold`).
3. **Dynamically handling skew in joins** — split one disproportionately large ("skewed") join partition into several smaller tasks so the query isn't held hostage by a single straggler task.

The key configs all live under the `spark.sql.adaptive.*` prefix. Let's inspect them.

In [0]:
# Inspect the key AQE-related settings on this cluster.
# AQE is enabled by default on Databricks; these are the knobs that control it.
aqe_configs = [
    "spark.sql.adaptive.enabled",                                   # master switch
    "spark.sql.adaptive.coalescePartitions.enabled",                # optimization 1
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",              # target partition size
    "spark.sql.adaptive.autoBroadcastJoinThreshold",                # optimization 2
    "spark.sql.adaptive.skewJoin.enabled",                          # optimization 3
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",            # skew detection factor
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",   # absolute skew threshold
]

for c in aqe_configs:
    try:
        print(f"{c:<58} = {spark.conf.get(c, '<not set>')}")
    except Exception:
        # On serverless (Spark Connect) compute, Databricks blocks reading/writing
        # most spark.sql.adaptive.* knobs from the notebook -- AQE is on and managed
        # server-side. On classic cluster compute the values would print here.
        print(f"{c:<58} = <managed server-side on this compute>")

spark.sql.adaptive.enabled                                 = <managed server-side on this compute>
spark.sql.adaptive.coalescePartitions.enabled              = <managed server-side on this compute>
spark.sql.adaptive.advisoryPartitionSizeInBytes            = <managed server-side on this compute>
spark.sql.adaptive.autoBroadcastJoinThreshold              = <managed server-side on this compute>
spark.sql.adaptive.skewJoin.enabled                        = <managed server-side on this compute>
spark.sql.adaptive.skewJoin.skewedPartitionFactor          = <managed server-side on this compute>
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = <managed server-side on this compute>


## Build a deliberately skewed dataset

To *see* AQE do something, we need data where the classic static planner would struggle: 2 million orders where **40% of all rows belong to a single `customer_id`**. One shuffle partition holding ~800k rows while the rest hold a few thousand is a textbook skew scenario.

In [0]:
from pyspark.sql import functions as F

# Small dimension table: 500 customers
customers = (
    spark.range(500)
    .withColumnRenamed("id", "customer_id")
    .withColumn("region", F.expr("concat('region-', cast(customer_id % 10 as string))"))
)

# Fact table: 2M orders, with heavy skew -- customer_id = 1 gets 40% of all rows
orders = (
    spark.range(2_000_000)
    .withColumn(
        "customer_id",
        F.expr("CASE WHEN id % 5 = 0 THEN 1 ELSE 1 + (id % 499) END")
    )
    .withColumn("amount", F.expr("round(rand() * 500, 2)"))
)

# Confirm the skew: one customer_id has ~800k rows, the rest have ~3-4k
orders.groupBy("customer_id").count().orderBy(F.desc("count")).show(5, truncate=False)

+-----------+------+
|customer_id|count |
+-----------+------+
|1          |403207|
|5          |3208  |
|224        |3207  |
|30         |3207  |
|29         |3207  |
+-----------+------+
only showing top 5 rows


## Demo 1 — Skew-join optimization

We join `orders` (skewed) to `customers` on `customer_id`. Without AQE, the partition holding `customer_id = 1` would be a single huge task, and the query's runtime would be dominated by that one straggler.

With `spark.sql.adaptive.skewJoin.enabled = true`, Spark detects partitions whose size exceeds both `skewedPartitionFactor` × the median *and* `skewedPartitionThresholdInBytes`, and **splits them into multiple sub-partitions**, replicating the matching rows from the other side so the join result stays correct. Watch for the `SortMergeJoin` in the plan and note how the query finishes quickly despite the skew.

> **Note for this compute:** on serverless compute the `spark.sql.adaptive.*` knobs are managed by Databricks (reading/setting them from the notebook is blocked), and skew-join handling is **on by default**. To still force a shuffle-based join for the demo, the next cell uses the `merge` join hint instead of config changes.

In [0]:
# Force a shuffle-based join the DataFrame way: the MERGE join hint requests
# a SortMergeJoin even though the customers side is small. (On classic cluster
# compute you could instead set spark.sql.autoBroadcastJoinThreshold = -1.)
joined = orders.hint("merge").join(customers, "customer_id")

# The physical plan: SortMergeJoin wrapped in an AdaptiveSparkPlan isSubquery=false.
# The printed plan shows the *initial* strategy; the skew split happens at runtime,
# once real per-partition statistics are measured at the shuffle boundary.
joined.explain("formatted")

print(f"join result row count: {joined.count()}")

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   Project (18)
   +- SortMergeJoin Inner (17)
      :- ColumnarToRow (8)
      :  +- PhotonResultStage (7)
      :     +- PhotonSort (6)
      :        +- PhotonShuffleExchangeSource (5)
      :           +- PhotonShuffleMapStage (4)
      :              +- PhotonShuffleExchangeSink (3)
      :                 +- PhotonProject (2)
      :                    +- PhotonRange (1)
      +- ColumnarToRow (16)
         +- PhotonResultStage (15)
            +- PhotonSort (14)
               +- PhotonShuffleExchangeSource (13)
                  +- PhotonShuffleMapStage (12)
                     +- PhotonShuffleExchangeSink (11)
                        +- PhotonProject (10)
                           +- PhotonRange (9)


(1) PhotonRange
Output [1]: [id#10127L]
Arguments: Range (0, 2000000, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#10127L]
Arguments: [id#10127L, CASE WHEN ((id#10127L % 5) = 0) THEN 1 ELSE ((id#10127L % 49

## Demo 2 — Dynamically coalescing shuffle partitions

`spark.sql.shuffle.partitions` (default 200) decides how many partitions a shuffle writes. That number is chosen **before** Spark knows the actual output size of the previous stage. For small results you get 200 tiny tasks, each with scheduling overhead.

With `spark.sql.adaptive.coalescePartitions.enabled = true`, after the shuffle write completes, AQE looks at the *actual* bytes per partition and merges them until each combined partition is roughly `advisoryPartitionSizeInBytes` (default 64 MB). A small result collapses to a handful of tasks instead of 200.

In [0]:
# Classic static setting: 200 shuffle partitions for a tiny aggregation
spark.conf.set("spark.sql.shuffle.partitions", "200")
# (coalescePartitions is on by default; on this serverless compute it's managed server-side)

agg = (
    orders
    .groupBy("customer_id")
    .agg(F.count("*").alias("n_orders"), F.round(F.sum("amount"), 2).alias("total_amount"))
)

# Only 500 distinct customer_ids -> the post-shuffle stage is tiny.
# AQE coalesces the 200 written partitions down to very few reduce tasks.
agg.explain("formatted")

agg.orderBy(F.desc("n_orders")).show(5, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#10127L]
Arguments: Range (0, 2000000, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#10127L]
Arguments: [CASE WHEN ((id#10127L % 5) = 0) THEN 1 ELSE ((id#10127L % 499) + 1) END AS customer_id#10129L, round((rand(-9075808452684205747) * 500.0), 2) AS amount#10131]

(3) PhotonGroupingAgg
Input [2]: [customer_id#10129L, amount#10131]
Arguments: [customer_id#10129L], [partial_count(1) AS count#10160L, partial_sum(amount#10131) AS sum#10162], [count#10159L, sum#10161], [customer_id#10129L, count#10160L, sum#10162], false

(4) PhotonShuffleExchangeS

## Demo 3 — Dynamically switching join strategies

Suppose the static planner (working from estimates) chooses a shuffle-based join, but at runtime the **actual** row count after a filter turns out to be tiny. AQE can flip the join into a **broadcast hash join**: instead of shuffling both sides, it broadcasts the small side to every executor — often an order-of-magnitude speedup.

The join below is *initially* forced into a shuffle (sort-merge) join with the `merge` hint. At runtime, once AQE sees the actual sizes — the filtered `customers` slice is tiny — it can convert the join into a **broadcast hash join**. Compare this with the Demo 1 plan, where Databricks broadcast the whole small dimension up front.

In [0]:
# Force the initial plan to a shuffle (sort-merge) join with the MERGE hint.
# At runtime, AQE knows the TRUE size of the filtered small side and can convert
# the join to a broadcast hash join (spark.sql.adaptive.autoBroadcastJoinThreshold).
big_side = orders.filter("customer_id < 100")      # sizeable fact slice
small_side = customers.filter("customer_id < 5")   # tiny after filtering

big_side.hint("merge").join(small_side, "customer_id").explain("formatted")

print("done -- note the AdaptiveSparkPlan node; the Spark UI shows AQE's final plan after execution")

== Physical Plan ==
AdaptiveSparkPlan (21)
+- == Initial Plan ==
   Project (20)
   +- SortMergeJoin Inner (19)
      :- ColumnarToRow (9)
      :  +- PhotonResultStage (8)
      :     +- PhotonSort (7)
      :        +- PhotonShuffleExchangeSource (6)
      :           +- PhotonShuffleMapStage (5)
      :              +- PhotonShuffleExchangeSink (4)
      :                 +- PhotonFilter (3)
      :                    +- PhotonProject (2)
      :                       +- PhotonRange (1)
      +- ColumnarToRow (18)
         +- PhotonResultStage (17)
            +- PhotonSort (16)
               +- PhotonShuffleExchangeSource (15)
                  +- PhotonShuffleMapStage (14)
                     +- PhotonShuffleExchangeSink (13)
                        +- PhotonProject (12)
                           +- PhotonFilter (11)
                              +- PhotonRange (10)


(1) PhotonRange
Output [1]: [id#10127L]
Arguments: Range (0, 2000000, step=1, splits=8)

(2) PhotonProject
Inpu

## Summary & further reading

| AQE feature | Config | What it fixes |
| --- | --- | --- |
| Shuffle partition coalescing | `spark.sql.adaptive.coalescePartitions.enabled` + `advisoryPartitionSizeInBytes` | Too many tiny reduce tasks after a small shuffle |
| Join strategy switching | `spark.sql.adaptive.autoBroadcastJoinThreshold` | Wrong (expensive) join type chosen from bad estimates |
| Skew-join splitting | `spark.sql.adaptive.skewJoin.enabled` + `skewedPartitionFactor` | One giant partition straggling while other tasks finish |

**Where to watch AQE in action:** the Spark UI query plan — executed plans show `AdaptiveSparkPlan` as the root node with the final, rewritten plan underneath, plus runtime metrics per stage.

**Handy to know:**

* AQE only re-optimizes at **shuffle stage boundaries** — stages that end without a shuffle in between can't adapt.
* `spark.sql.adaptive.enabled` is the master switch (on by default on Databricks).
* On Databricks, AQE composes with Photon; the same concepts apply.
* If a query is already fast, don't tune these knobs — AQE's defaults are good.